# R3-04 · Agentes: herramientas, trazas y supervisión humana — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R3 · *IA Aplicada* · Semana 5 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Escribir **herramientas** (funciones de Python) que un agente puede usar.
2. Construir un **planificador de respaldo** por reglas y **validar** lo que propone el LLM.
3. Ejecutar paso a paso dejando una **traza** auditable.
4. Exigir **aprobación humana** antes de toda acción con efecto.
5. Cerrar el bucle del agente con un **límite de pasos**.

**Competencia de salida:** orquestar herramientas de datos con un LLM de forma segura, auditable y verificable sin red, manteniendo a una persona a cargo de las decisiones.

**Dato real:** PM2.5 diario de 14 estaciones SINCA 2020–2025 (rama R2) y matrícula escolar 2025 MINEDUC (rama R1).

**Entregable:** que las **6 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, re, json, unicodedata, urllib.request
import pandas as pd

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["aire_diario.csv", "estaciones.csv", "matricula.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

aire = pd.read_csv("aire_diario.csv", parse_dates=["fecha"])   # PM2.5 diario SINCA (rama R2)
estaciones = pd.read_csv("estaciones.csv")                      # dónde está cada estación
matricula = pd.read_csv("matricula.csv")                        # matrícula 2025 MINEDUC (rama R1)

def normalizar(texto):
    """minúsculas y sin tildes, para comparar textos escritos por personas."""
    t = unicodedata.normalize("NFKD", str(texto).lower())
    return "".join(c for c in t if not unicodedata.combining(c))

print("aire:", aire.shape, "|", aire["estacion"].nunique(), "estaciones,",
      aire["fecha"].min().date(), "a", aire["fecha"].max().date())
print("matrícula:", matricula.shape, "|", matricula["comuna"].nunique(), "comunas")

### El LLM detrás de una sola función

La misma `preguntar_llm(prompt, respaldo=...)` de R3-02 y R3-03. Sin `GEMINI_API_KEY` devuelve el
`respaldo` y el agente usa su **planificador de reglas**: todo el notebook funciona igual y ningún
chequeo depende de lo que conteste el modelo.

> Si en tu entorno falta el SDK de Google, instálalo con `!pip install -q google-genai`.

In [ ]:
# ── preguntar_llm(): el LLM (Gemini) detrás de UNA función ─────────────────────
# Cambiar de proveedor o de modelo = cambiar solo esta celda.
# En Colab: 🔑 (Secretos) → agrega GEMINI_API_KEY (gratis en aistudio.google.com).
import os

MODELO = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite")

def _clave_gemini():
    try:                                   # Colab: secreto del notebook
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:                      # local: variable de entorno
        return os.environ.get("GEMINI_API_KEY")

def preguntar_llm(prompt, respaldo="[respuesta de respaldo — configura GEMINI_API_KEY para usar el LLM en vivo]"):
    """Envía el prompt a Gemini y devuelve el texto. Sin clave o sin red, devuelve
    `respaldo` para que el notebook siga corriendo (y se pueda verificar en CI)."""
    clave = _clave_gemini()
    if not clave:
        return respaldo
    try:
        from google import genai
        cliente = genai.Client(api_key=clave)
        return cliente.models.generate_content(model=MODELO, contents=prompt).text
    except Exception as e:
        print("⚠️ No se pudo llamar al LLM:", e)
        return respaldo

def get_llm():
    """Compatibilidad: devuelve una función generate(prompt)->str."""
    return preguntar_llm

print("Modelo:", MODELO, "| modo:", "EN VIVO" if _clave_gemini() else "RESPALDO (planificador de reglas)")

## ¿Qué es un agente?

Un programa normal sigue una receta fija. Un **agente** recibe una petición en lenguaje natural y
**decide qué hacer**, paso a paso:

1. **Planificar:** ¿qué herramienta uso ahora, y con qué argumentos? (aquí opina el LLM)
2. **Actuar:** ejecutar esa herramienta — una **función de Python** que tú escribiste.
3. **Observar:** mirar el resultado y volver a 1… hasta decidir que terminó.

Este módulo **cierra el arco de las tres ramas**: las herramientas consultan la **matrícula escolar**
(el dato ancla de R1) y la **calidad del aire** (el de R2); el LLM (R3) solo **orquesta**. Las
cifras nunca las inventa el modelo: salen de pandas.

### Las cuatro reglas de un agente público

| Regla | Por qué | Dónde la construyes |
|---|---|---|
| El LLM **propone**, el código **valida** | Un LLM puede pedir una herramienta que no existe o inventar argumentos | Ejercicio 3 |
| **Traza** de cada paso | Poder responder «¿por qué el sistema hizo esto?» | Ejercicio 4 |
| **Aprobación humana** antes de toda acción con efecto | Un agente no envía, borra ni paga nada por su cuenta | Ejercicio 5 |
| **Límite de pasos** | Evita bucles infinitos (y cuentas infinitas) | Ejercicio 6 |

> Siempre hay un **planificador de respaldo por reglas**, determinista: el agente funciona sin red
> y se puede verificar. Es también lo que usa cuando el LLM responde algo inválido.

## Las herramientas (y el «menú» del agente)

Tres herramientas de **consulta** ya están listas — `pm25_promedio`, `matricula_comuna` — y una
**acción con efecto**: `enviar_alerta`, que simula el envío de un correo (lo deja en
`BANDEJA_SALIDA`). Falta una: la vas a escribir tú.

In [ ]:
NORMA_PM25_DIARIA = 50   # µg/m³: valor de la norma primaria diaria de PM2.5 en Chile (DS 12/2011 MMA)

def pm25_promedio(estacion, anio):
    """PM2.5 promedio de un año (µg/m³) en una estación SINCA."""
    d = aire[(aire["estacion"] == estacion) & (aire["fecha"].dt.year == anio)]["pm25"].dropna()
    if d.empty:
        raise ValueError(f"no hay datos de PM2.5 para «{estacion}» en {anio}")
    return {"estacion": estacion, "anio": anio, "pm25_promedio": round(float(d.mean()), 1),
            "dias_con_dato": int(d.size)}

def matricula_comuna(comuna):
    """Establecimientos y estudiantes matriculados (2025) en una comuna."""
    d = matricula[matricula["comuna"].map(normalizar) == normalizar(comuna)]
    if d.empty:
        raise ValueError(f"no conozco la comuna «{comuna}»")
    return {"comuna": d["comuna"].iloc[0], "establecimientos": int(len(d)),
            "estudiantes": int(d["mat_total"].sum())}

BANDEJA_SALIDA = []   # simula el correo institucional: aquí «llegan» las alertas enviadas

def enviar_alerta(destinatario, mensaje):
    """ACCIÓN CON EFECTO: envía (simulado) una alerta a un destinatario."""
    BANDEJA_SALIDA.append({"para": destinatario, "mensaje": mensaje})
    return {"enviado_a": destinatario}

## 1. Escribe una herramienta: días sobre la norma

Una herramienta es una función normal de pandas con tres virtudes: hace **una** cosa, recibe argumentos **simples** (texto, números) y **avisa con un error claro** si no puede. Escribe `dias_sobre_norma(estacion, anio)`: cuántos días del año el PM2.5 superó 50 µg/m³ (el valor de la norma diaria) y con cuántos días con dato se calculó.

> Ojo con la interpretación: legalmente la norma diaria se evalúa con el **percentil 98** del año, así que «días sobre 50» es un indicador de episodios, no un dictamen de incumplimiento.

### ✍️ Ejercicio 1 — Escribe `dias_sobre_norma`

In [ ]:
def dias_sobre_norma(estacion, anio, umbral=NORMA_PM25_DIARIA):
    """Días de un año con PM2.5 sobre el umbral (por defecto, 50 µg/m³) en una estación."""
    d = aire[(aire["estacion"] == estacion) & (aire["fecha"].dt.year == anio)]["pm25"].dropna()
    if d.empty:
        raise ValueError(f"no hay datos de PM2.5 para «{estacion}» en {anio}")
    return {"estacion": estacion, "anio": anio, "dias_sobre_norma": int((d > umbral).sum()),
            "dias_con_dato": int(d.size)}

dias_sobre_norma("Cerro Navia", 2024)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    r = dias_sobre_norma("Cerro Navia", 2024)
    d = aire[(aire["estacion"] == "Cerro Navia") & (aire["fecha"].dt.year == 2024)]["pm25"].dropna()
    assert isinstance(r, dict), "la herramienta debe devolver un diccionario"
    assert r["dias_sobre_norma"] == int((d > 50).sum()), "cuenta los días con pm25 > umbral"
    assert r["dias_con_dato"] == int(d.size), "dias_con_dato = cantidad de días con medición"
    try:
        dias_sobre_norma("Estación Inventada", 2024)
        raise AssertionError("con una estación inexistente debe lanzar ValueError")
    except ValueError:
        pass
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'return {"estacion": estacion, "anio": anio, "dias_sobre_norma": int((d > umbral).sum()), "dias_con_dato": int(d.size)}.')
    print("   Detalle:", e)

### El menú: `HERRAMIENTAS`

El agente solo puede usar lo que está en este diccionario. Cada entrada declara sus argumentos y
si tiene **efecto** (⚠️). Ese simple `True/False` es el que, en el ejercicio 5, activa la
aprobación humana. **Permisos mínimos:** si una herramienta no está en el menú, el agente no la tiene.

In [ ]:
# El «menú» del agente: qué herramientas existen, qué argumentos piden y si tienen EFECTO
# (cambian algo en el mundo). El agente no puede usar nada que no esté aquí.
HERRAMIENTAS = {
    "pm25_promedio":    {"funcion": pm25_promedio,    "args": ["estacion", "anio"], "efecto": False,
                         "descripcion": "PM2.5 promedio anual (µg/m³) de una estación SINCA, años 2020-2025."},
    "dias_sobre_norma": {"funcion": dias_sobre_norma, "args": ["estacion", "anio"], "efecto": False,
                         "descripcion": "Días del año con PM2.5 sobre 50 µg/m³ en una estación, años 2020-2025."},
    "matricula_comuna": {"funcion": matricula_comuna, "args": ["comuna"], "efecto": False,
                         "descripcion": "Establecimientos y estudiantes matriculados en 2025 en una comuna."},
    "enviar_alerta":    {"funcion": enviar_alerta,    "args": ["destinatario", "mensaje"], "efecto": True,
                         "descripcion": "ENVÍA una alerta. Acción con efecto: requiere aprobación humana."},
}
for nombre, d in HERRAMIENTAS.items():
    print(f"{'⚠️ ' if d['efecto'] else '   '}{nombre}({', '.join(d['args'])})")

### Entender la petición

Para el planificador de reglas necesitamos sacar del texto la **estación**, el **año** y la
**comuna**. Fíjate en el último ejemplo: «Temuco» tiene **dos** estaciones (Las Encinas y Nielol),
así que no se usa como alias — es ambiguo, y es mejor no adivinar.

In [ ]:
# Del texto de la petición a los argumentos: estación, año y comuna.
ALIAS_ESTACION = {}
comunas_con_estacion = estaciones["comuna"].map(normalizar).value_counts()
for est, com in zip(estaciones["estacion"], estaciones["comuna"]):
    ALIAS_ESTACION[normalizar(re.sub(r"\s*\(.*\)", "", est))] = est   # «El Bosque (Acreditada)» → «el bosque»
    if comunas_con_estacion[normalizar(com)] == 1:                      # la comuna sirve de alias si no es ambigua
        ALIAS_ESTACION.setdefault(normalizar(com), est)

def extraer_estacion(texto):
    t = normalizar(texto)
    for alias in sorted(ALIAS_ESTACION, key=len, reverse=True):
        if re.search(rf"\b{re.escape(alias)}\b", t):
            return ALIAS_ESTACION[alias]
    return None

def extraer_anio(texto):
    m = re.search(r"\b(20\d\d)\b", texto)
    return int(m.group(1)) if m else None

COMUNAS = sorted(matricula["comuna"].unique(), key=len, reverse=True)
def extraer_comuna(texto):
    t = normalizar(texto)
    for c in COMUNAS:
        if re.search(rf"\b{re.escape(normalizar(c))}\b", t):
            return c
    return None

print(extraer_estacion("calidad del aire en Rancagua"), "|", extraer_anio("en 2023"), "|",
      extraer_comuna("estudiantes de Ñuñoa"), "|", extraer_estacion("aire en Temuco"))

## 2. El planificador de respaldo (reglas)

El planificador recibe la petición y la **traza** (lo que ya se hizo) y devuelve el **siguiente paso**: `{"herramienta": ..., "args": {...}}`, o `{"herramienta": "fin", ...}` si ya terminó.

La parte de «¿ya tenemos el dato? → ¿piden avisar?» está escrita. Completa la decisión del **primer paso**, en este orden:

1. Si menciona **norma** o **episodio** → `dias_sobre_norma` (estación y año).
2. Si menciona **pm2**, **aire** o **contaminacion** → `pm25_promedio` (estación y año).
3. Si menciona **matricula**, **estudiantes** o **colegios** → `matricula_comuna` (comuna).
4. Si nada calza → `fin` (mejor no hacer nada que inventar).

`t` ya viene normalizado (minúsculas, sin tildes).

### ✍️ Ejercicio 2 — Completa `planificar_reglas`

In [ ]:
def planificar_reglas(peticion, traza):
    """Planificador de RESPALDO, determinista: decide el siguiente paso con reglas simples."""
    t = normalizar(peticion)
    hechos = [p for p in traza if p["estado"] == "ok"]
    if any(p["estado"] in ("error", "bloqueado_por_guarda") for p in traza):   # algo falló: no insistir
        return {"herramienta": "fin", "args": {}}
    if hechos:                                               # ya tenemos el dato
        quiere_avisar = any(w in t for w in ["avisa", "alerta", "notifica"])
        ya_propuesta = any(p["herramienta"] == "enviar_alerta" for p in traza)
        if quiere_avisar and not ya_propuesta:
            return {"herramienta": "enviar_alerta",
                    "args": {"destinatario": "SEREMI de Medio Ambiente",
                             "mensaje": f"Resultado de la consulta: {hechos[-1]['resultado']}"}}
        return {"herramienta": "fin", "args": {}}
    # Primer paso: ¿qué dato hay que consultar?
    if "norma" in t or "episodio" in t:
        return {"herramienta": "dias_sobre_norma",
                "args": {"estacion": extraer_estacion(peticion), "anio": extraer_anio(peticion)}}
    if "pm2" in t or "aire" in t or "contaminacion" in t:
        return {"herramienta": "pm25_promedio",
                "args": {"estacion": extraer_estacion(peticion), "anio": extraer_anio(peticion)}}
    if "matricula" in t or "estudiantes" in t or "colegios" in t:
        return {"herramienta": "matricula_comuna", "args": {"comuna": extraer_comuna(peticion)}}
    return {"herramienta": "fin", "args": {}}

for p in ["¿Cuántos días sobre la norma tuvo Cerro Navia en 2024?",
          "¿Cuál fue el PM2.5 promedio de Rancagua en 2023?",
          "¿Cuántos estudiantes hay en Temuco?", "¿Cómo está el tráfico hoy?"]:
    print(planificar_reglas(p, []), "←", p)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    casos = {
        "¿Cuántos días sobre la norma tuvo Cerro Navia en 2024?":
            {"herramienta": "dias_sobre_norma", "args": {"estacion": "Cerro Navia", "anio": 2024}},
        "¿Cuál fue el PM2.5 promedio de Rancagua en 2023?":
            {"herramienta": "pm25_promedio", "args": {"estacion": "Rancagua I", "anio": 2023}},
        "¿Cuántos estudiantes hay en Temuco?":
            {"herramienta": "matricula_comuna", "args": {"comuna": "Temuco"}},
        "¿Cómo está el tráfico hoy?": {"herramienta": "fin", "args": {}},
    }
    for peticion, esperado in casos.items():
        obtenido = planificar_reglas(peticion, [])
        assert obtenido == esperado, f"«{peticion}» → esperaba {esperado}, obtuve {obtenido}"
    ya = [{"herramienta": "dias_sobre_norma", "estado": "ok", "resultado": {"dias_sobre_norma": 56}}]
    assert planificar_reglas("días sobre la norma en Cerro Navia 2024 y avisa a la SEREMI", ya)["herramienta"] == "enviar_alerta"
    assert planificar_reglas("días sobre la norma en Cerro Navia 2024", ya)["herramienta"] == "fin"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'if "norma" in t or "episodio" in t: return {"herramienta": "dias_sobre_norma", "args": {...}}; igual para las otras dos reglas.')
    print("   Detalle:", e)

## 3. El LLM propone, el código valida

Cuando hay clave, el planificador le muestra al LLM el menú y le pide el siguiente paso **en JSON**. Pero lo que vuelve es **texto**: puede venir entre ```` ``` ````, pedir una herramienta que no existe («borrar_base»), olvidar un argumento… o no ser JSON. `interpretar_plan` es la aduana: devuelve un plan **válido** o `None` (y entonces decide el planificador de reglas).

La extracción del JSON ya está; completa la **validación**: la herramienta debe estar en `HERRAMIENTAS` y los `args` deben ser un diccionario con **exactamente** los argumentos que declara el menú.

### ✍️ Ejercicio 3 — Completa la validación de `interpretar_plan`

In [ ]:
def interpretar_plan(texto):
    """Convierte la respuesta del LLM en un plan VÁLIDO, o None si no se puede confiar en ella."""
    m = re.search(r"\{.*\}", texto or "", flags=re.S)        # el JSON puede venir entre ```
    if not m:
        return None
    try:
        plan = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    herramienta = plan.get("herramienta")
    if herramienta == "fin":
        return {"herramienta": "fin", "args": {}}
    if herramienta not in HERRAMIENTAS:
        return None
    args = plan.get("args")
    if not isinstance(args, dict) or set(args) != set(HERRAMIENTAS[herramienta]["args"]):
        return None
    return {"herramienta": herramienta, "args": args}

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    ok = interpretar_plan('```json\n{"herramienta": "pm25_promedio", "args": {"estacion": "Nielol", "anio": 2022}}\n```')
    assert ok == {"herramienta": "pm25_promedio", "args": {"estacion": "Nielol", "anio": 2022}}, "un JSON válido entre ``` debe aceptarse"
    assert interpretar_plan('{"herramienta": "borrar_base_de_datos", "args": {}}') is None, "herramienta inexistente → None"
    assert interpretar_plan('{"herramienta": "pm25_promedio", "args": {"estacion": "Nielol"}}') is None, "falta un argumento → None"
    assert interpretar_plan('{"herramienta": "matricula_comuna", "args": {"comuna": "Arica", "borrar": true}}') is None, "argumento de más → None"
    assert interpretar_plan("Claro, con gusto te ayudo.") is None, "texto sin JSON → None"
    assert interpretar_plan('{"herramienta": "fin"}') == {"herramienta": "fin", "args": {}}
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'if herramienta not in HERRAMIENTAS: return None  ·  if not isinstance(args, dict) or set(args) != set(HERRAMIENTAS[herramienta]["args"]): return None.')
    print("   Detalle:", e)

### El planificador principal: LLM con respaldo

Esta función arma el prompt (menú + estaciones válidas + pasos ya hechos), se lo envía al LLM,
pasa la respuesta por tu `interpretar_plan` y, si no sirve —o no hay clave—, usa
`planificar_reglas`. Cada plan queda marcado con su **origen** (`llm` o `reglas`) para la traza.

In [ ]:
def describir_herramientas():
    return "\n".join(f"- {n}({', '.join(d['args'])}): {d['descripcion']}" for n, d in HERRAMIENTAS.items())

def planificar(peticion, traza):
    prompt = ("Eres el planificador de un agente de datos públicos de Chile.\n"
              f"Herramientas disponibles:\n{describir_herramientas()}\n"
              f"Estaciones válidas: {', '.join(estaciones['estacion'])}.\n\n"
              f"Petición: {peticion}\n"
              f"Pasos ya realizados: {json.dumps(traza, ensure_ascii=False, default=str)}\n\n"
              'Decide SOLO el siguiente paso. Responde únicamente un JSON: '
              '{"herramienta": "...", "args": {...}}, o {"herramienta": "fin", "args": {}} si ya se puede responder.')
    plan = interpretar_plan(preguntar_llm(prompt, respaldo=""))
    if plan is None:
        plan = dict(planificar_reglas(peticion, traza), origen="reglas")
    else:
        plan["origen"] = "llm"
    return plan

print(planificar("¿Cuántos días sobre la norma tuvo Cerro Navia en 2024?", []))

## 4. Ejecutar con traza

Cada paso que da el agente queda **registrado**: número, herramienta, argumentos, quién lo propuso, resultado y estado. Si la herramienta falla (estación inexistente, año sin datos), el agente **no se cae**: registra el error y sigue. La traza es el **expediente** del agente.

### ✍️ Ejercicio 4 — Completa `ejecutar_paso`

In [ ]:
def ejecutar_paso(plan, traza):
    """Ejecuta UNA herramienta y deja el registro en la traza (sin caerse si falla)."""
    nombre, args = plan["herramienta"], plan["args"]
    registro = {"paso": len(traza) + 1, "herramienta": nombre, "args": args,
                "origen": plan.get("origen", "manual")}
    try:
        registro["resultado"] = HERRAMIENTAS[nombre]["funcion"](**args)
        registro["estado"] = "ok"
    except Exception as e:
        registro["resultado"] = f"{type(e).__name__}: {e}"
        registro["estado"] = "error"
    traza.append(registro)
    return registro

traza_demo = []
ejecutar_paso({"herramienta": "pm25_promedio", "args": {"estacion": "Nielol", "anio": 2022}}, traza_demo)
ejecutar_paso({"herramienta": "pm25_promedio", "args": {"estacion": None, "anio": 2022}}, traza_demo)
pd.DataFrame(traza_demo)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    t = []
    r1 = ejecutar_paso({"herramienta": "pm25_promedio", "args": {"estacion": "Nielol", "anio": 2022}}, t)
    assert r1["estado"] == "ok" and r1["resultado"] == pm25_promedio("Nielol", 2022), "un paso válido debe quedar 'ok' con su resultado"
    r2 = ejecutar_paso({"herramienta": "pm25_promedio", "args": {"estacion": "No existe", "anio": 2022}}, t)
    assert r2["estado"] == "error" and "ValueError" in r2["resultado"], "un paso que falla debe quedar 'error', sin caerse"
    assert [p["paso"] for p in t] == [1, 2], "cada paso queda numerado en la traza"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'try: registro["resultado"] = HERRAMIENTAS[nombre]["funcion"](**args); registro["estado"] = "ok"  /  except Exception as e: ... "error".')
    print("   Detalle:", e)

## 5. Supervisión humana: nada con efecto sin aprobación

`enviar_alerta` **cambia el mundo**: llega un correo a la SEREMI. Ningún agente público debería hacer eso solo. Regla: si la herramienta tiene `efecto=True`, se ejecuta **solo si** un `aprobador` (una persona) dice que sí. Y si no hay aprobador, la respuesta es **no** (*denegar por defecto*). La negativa también queda en la traza.

### ✍️ Ejercicio 5 — Completa `ejecutar_supervisado`

In [ ]:
def ejecutar_supervisado(plan, traza, aprobador=None):
    """Si la herramienta tiene EFECTO, solo se ejecuta con aprobación humana explícita."""
    if HERRAMIENTAS[plan["herramienta"]]["efecto"]:
        if aprobador is None or not aprobador(plan):         # por defecto: NO
            registro = {"paso": len(traza) + 1, "herramienta": plan["herramienta"],
                        "args": plan["args"], "origen": plan.get("origen", "manual"),
                        "resultado": "acción NO ejecutada: sin aprobación humana",
                        "estado": "rechazado"}
            traza.append(registro)
            return registro
        registro = ejecutar_paso(plan, traza)
        registro["aprobado_por_humano"] = True
        return registro
    return ejecutar_paso(plan, traza)

alerta = {"herramienta": "enviar_alerta", "args": {"destinatario": "SEREMI", "mensaje": "prueba"}}
print(ejecutar_supervisado(alerta, [], aprobador=None)["estado"], "| bandeja:", len(BANDEJA_SALIDA))

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    antes = len(BANDEJA_SALIDA)
    alerta = {"herramienta": "enviar_alerta", "args": {"destinatario": "SEREMI", "mensaje": "prueba"}}
    t = []
    assert ejecutar_supervisado(alerta, t, aprobador=None)["estado"] == "rechazado", "sin aprobador: no se ejecuta"
    assert ejecutar_supervisado(alerta, t, aprobador=lambda p: False)["estado"] == "rechazado", "si la persona dice no: no se ejecuta"
    assert len(BANDEJA_SALIDA) == antes, "¡se envió una alerta sin aprobación!"
    assert ejecutar_supervisado(alerta, t, aprobador=lambda p: True)["estado"] == "ok", "con aprobación sí se ejecuta"
    assert len(BANDEJA_SALIDA) == antes + 1
    assert ejecutar_supervisado({"herramienta": "matricula_comuna", "args": {"comuna": "Arica"}}, t)["estado"] == "ok", "las consultas no requieren aprobación"
    assert len(t) == 4
    BANDEJA_SALIDA.pop()   # limpiamos la alerta de prueba
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'if aprobador is None or not aprobador(plan): registro = {..., "estado": "rechazado"}; traza.append(registro); return registro.')
    print("   Detalle:", e)

### La respuesta final

Con la traza lista, el LLM **redacta** la respuesta usando **solo** los resultados de las
herramientas (no puede agregar cifras). Sin clave, el respaldo es un resumen literal de la traza.

In [ ]:
def redactar_respuesta(peticion, traza, estado):
    resumen = " | ".join(f"{p['herramienta']} [{p['estado']}] → {p['resultado']}" for p in traza)
    respaldo = f"({estado}) {resumen}" if traza else "No tengo una herramienta para responder eso."
    if not any(p["estado"] == "ok" for p in traza):
        return respaldo
    prompt = ("Redacta en 2 o 3 frases, en español claro, la respuesta a esta petición usando SOLO "
              "estos resultados. No agregues cifras que no estén ahí. Si una acción fue rechazada, dilo.\n"
              f"Petición: {peticion}\nResultados: {resumen}")
    return preguntar_llm(prompt, respaldo=respaldo)

## 6. El bucle del agente, con límite de pasos

Ahora todo junto: planificar → (supervisar) → ejecutar → volver a planificar… hasta que el planificador diga `fin` **o** se acabe el presupuesto de `max_pasos`. Ese límite existe porque los agentes con LLM a veces entran en bucles («consulto otra vez, por si acaso»), y cada vuelta cuesta tiempo y dinero. Si se alcanza el límite, el estado lo dice.

### ✍️ Ejercicio 6 — Completa el bucle de `agente`

In [ ]:
def agente(peticion, planificador=None, aprobador=None, max_pasos=4):
    planificador = planificador or planificar
    traza, estado = [], "limite_de_pasos"
    for _ in range(max_pasos):
        plan = planificador(peticion, traza)
        if plan["herramienta"] == "fin":
            estado = "terminado"
            break
        ejecutar_supervisado(plan, traza, aprobador)
    return {"estado": estado, "traza": traza, "respuesta": redactar_respuesta(peticion, traza, estado)}

r = agente("¿Cuántos días sobre la norma tuvo Cerro Navia en 2024? Avisa a la SEREMI.",
           planificador=planificar_reglas, aprobador=None)
print(r["estado"]); print(pd.DataFrame(r["traza"])[["paso", "herramienta", "estado", "resultado"]])

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    antes = len(BANDEJA_SALIDA)
    r = agente("¿Cuántos días sobre la norma tuvo Cerro Navia en 2024? Avisa a la SEREMI.",
               planificador=planificar_reglas, aprobador=lambda p: False)
    assert r["estado"] == "terminado", "el agente debería terminar por sí mismo"
    assert [p["herramienta"] for p in r["traza"]] == ["dias_sobre_norma", "enviar_alerta"]
    assert [p["estado"] for p in r["traza"]] == ["ok", "rechazado"], "la alerta debe quedar rechazada (la persona dijo no)"
    assert len(BANDEJA_SALIDA) == antes
    def planificador_terco(peticion, traza):        # simula un LLM atrapado en un bucle
        return {"herramienta": "pm25_promedio", "args": {"estacion": "Nielol", "anio": 2022}}
    r2 = agente("cualquier cosa", planificador=planificador_terco, max_pasos=3)
    assert r2["estado"] == "limite_de_pasos" and len(r2["traza"]) == 3, "el límite de pasos debe cortar el bucle"
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'if plan["herramienta"] == "fin": estado = "terminado"; break  — y luego ejecutar_supervisado(plan, traza, aprobador).')
    print("   Detalle:", e)

### El agente completo, en acción

Ahora con el planificador principal (`planificar`: LLM si hay clave, reglas si no). Fíjate en la
columna `origen` de la traza y en que la alerta **no** sale, porque nadie la aprobó.

> 🧑‍⚖️ En Colab, prueba la aprobación de verdad: `agente("...avisa a la SEREMI", aprobador=aprobador_consola)`
> te mostrará la acción y te preguntará si la apruebas.

In [ ]:
def aprobador_consola(plan):
    print("⚠️ El agente quiere ejecutar:", plan["herramienta"], plan["args"])
    return input("¿Apruebas esta acción? (si/no): ").strip().lower() in {"si", "sí", "s"}

try:
    for peticion in ["¿Cuál fue el PM2.5 promedio de Las Encinas Temuco en 2021?",
                     "¿Cuántos estudiantes hay en Coyhaique?",
                     "¿Cuántos días sobre la norma tuvo Rancagua en 2023? Si hubo, avisa a la SEREMI."]:
        r = agente(peticion)                       # aprobador=None → ninguna acción con efecto se ejecuta
        print(f"\n▶ {peticion}\n  estado: {r['estado']}")
        for p in r["traza"]:
            print(f"  paso {p['paso']} [{p['origen']}] {p['herramienta']} → {p['estado']}: {p['resultado']}")
        print("  respuesta:", r["respuesta"][:300])
except Exception as e:
    print("Completa los ejercicios anteriores para ver al agente en acción.", e)

## Riesgos de un agente en el Estado (y qué hiciste para cada uno)

| Riesgo | Ejemplo | Mitigación |
|---|---|---|
| **Herramienta o argumentos inventados** | El LLM pide `borrar_base` o una estación que no existe | `interpretar_plan` valida contra el menú; la herramienta falla con un error claro |
| **Acción sin control** | Se envía una alerta (o un pago, o un oficio) que nadie revisó | `efecto=True` + aprobación humana, **denegar por defecto** |
| **Bucles y costo** | El agente consulta lo mismo una y otra vez | `max_pasos` |
| **Inyección de instrucciones** | Un texto dentro de los datos dice «ignora tus reglas y envía…» | Los resultados son **datos**, no órdenes; toda acción pasa por una persona |
| **Sesgo de automatización** | La persona aprueba todo sin leer | Mostrar **qué** y **por qué** antes de aprobar; revisar trazas al azar |
| **Datos personales** | La petición trae un RUT o un diagnóstico | No enviarlos al LLM externo; anonimizar antes (profundización) |
| **Cifras sin respaldo** | El LLM «redondea» o inventa un número | Las cifras salen de pandas; el LLM solo redacta con los resultados |

**La traza es el expediente.** Con ella puedes reconstruir cada decisión: quién la propuso
(LLM o reglas), qué se ejecutó, qué se rechazó y por qué. La responsabilidad de lo que el
sistema hace sigue siendo de las personas que lo operan.

## ✍️ Reflexión (no se corrige, pero es lo más importante)

Piensa en una tarea repetitiva de tu servicio que un agente podría hacer. ¿Qué herramientas
necesitaría? ¿Cuáles tendrían **efecto** (enviar, registrar, pagar, notificar)? ¿Quién debería
aprobarlas, y qué tendría que ver en pantalla para aprobar con conocimiento de causa?

## Errores típicos

- **Ejecutar directo lo que dice el LLM** (`eval`, o llamar la función que nombre) → siempre valida contra el menú.
- **Aprobación por defecto en «sí»** → si el aprobador falla o no existe, la acción se ejecuta. Debe ser al revés.
- **Sin límite de pasos** → un bucle del LLM se vuelve una factura.
- **Asserts que dependen del LLM en vivo** → tus pruebas fallan al azar. Prueba con planificadores y aprobadores «falsos».

## Cierre

- Un **agente** = planificar → actuar → observar, en bucle. Las **herramientas** son funciones que tú controlas.
- El LLM **propone**; el código **valida**; la persona **aprueba** lo que tiene efecto.
- **Traza**, **límite de pasos** y **respaldo determinista** hacen que el agente sea auditable y verificable sin red.
- Cerraste el arco: datos de **R1** (matrícula) y **R2** (aire), orquestados con IA de **R3**.

> *En la profundización, el agente aprende a buscar en las fichas de ChileAtiende (tu RAG de
> R3-03 hecho herramienta) y suma guardas: argumentos válidos, calidad del dato y privacidad.
> Después viene el capstone.*